In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.cluster import KMeans

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, mean_absolute_error, r2_score
)
try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error
    def root_mean_squared_error(y_true, y_pred):
        return mean_squared_error(y_true, y_pred) ** 0.5

from sklearn.metrics import silhouette_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.datasets import fetch_california_housing
from sklearn.pipeline import make_pipeline


In [ ]:
def charger_immobilier():
  
    data = fetch_california_housing()
    X = data.data
    y = data.target 

    print(f"California Housing : {X.shape[0]} lignes, {X.shape[1]} variables")
    print(f"Variables : {list(data.feature_names)}")
    print(f"Cible : prix médian en centaines de milliers de $")
    print(f"  min={y.min():.2f}, max={y.max():.2f}, moyenne={y.mean():.2f}")

    return X, y


def evaluer_regression(nom_modele, modele, X_train, X_test, y_train, y_test):
  
    modele.fit(X_train, y_train)
    y_pred = modele.predict(X_test)

    r2   = r2_score(y_test, y_pred)
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)

    print(f"{nom_modele:<22} : R2={r2:.2f}  MAE={mae:.2f}  RMSE={rmse:.2f}")
    return {"r2": r2, "mae": mae, "rmse": rmse}


X_immo, y_immo = charger_immobilier()

X_tr_i, X_te_i, y_tr_i, y_te_i = train_test_split(
    X_immo, y_immo, test_size=0.2, random_state=42
)
scaler_i = StandardScaler()
X_tr_is = scaler_i.fit_transform(X_tr_i)
X_te_is = scaler_i.transform(X_te_i)

print("\n--- Résultats ---")
res_lr = evaluer_regression(
    "LinearRegression",
    LinearRegression(), X_tr_is, X_te_is, y_tr_i, y_te_i
)
res_rf = evaluer_regression(
    "RandomForest",
    RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    X_tr_is, X_te_is, y_tr_i, y_te_i
)


In [ ]:


data_immo = fetch_california_housing()
lr_interp = LinearRegression().fit(X_tr_is, y_tr_i)

print("Coefficients de la régression linéaire :")
print("(+coef = augmente le prix | -coef = baisse le prix)")
print()
for nom, poids in zip(data_immo.feature_names, lr_interp.coef_):
    print(f"  {nom:>15} : {poids:+.3f}")


In [ ]:

lr_small = LinearRegression()
lr_small.fit(X_tr_is[:100], y_tr_i[:100])
r2_small = r2_score(y_te_i, lr_small.predict(X_te_is))
print(f"R2 avec 100 lignes seulement : {r2_small:.3f}")
print(f"R2 avec le dataset complet   : {res_lr['r2']:.3f}")



quartier_fictif = np.zeros((1, X_immo.shape[1]))
quartier_fictif[0, 0] = 0.0  
quartier_fictif[0, 4] = 9000 

quartier_fictif_scaled = scaler_i.transform(quartier_fictif)
prix_predit = lr_interp.predict(quartier_fictif_scaled)[0]

print(f"Quartier fictif (revenu=0, pop=9000) → prix prédit : {prix_predit:.2f} (×100k$)")


In [ ]:
def charger_airbnb(url_csv):
  
    try:
        df = pd.read_csv(url_csv, low_memory=False)
    except Exception as e:
        print(f"⚠️  Impossible de charger depuis l'URL : {e}")
        print("→ On génère des données synthétiques pour la démonstration.")
        return None

    colonnes_utiles = [
        "price", "minimum_nights", "number_of_reviews", "availability_365"
    ]
    colonnes_dispo = [c for c in colonnes_utiles if c in df.columns]

    if not colonnes_dispo:
        return None

    df_clean = df[colonnes_dispo].copy()

    if "price" in df_clean.columns and df_clean["price"].dtype == object:
        df_clean["price"] = df_clean["price"].str.replace("[$,]", "", regex=True)
        df_clean["price"] = pd.to_numeric(df_clean["price"], errors="coerce")

    df_clean = df_clean.dropna()
    if "price" in df_clean.columns:
        df_clean = df_clean[df_clean["price"] > 0]
        df_clean = df_clean[df_clean["price"] < 1000]  

    print(f"✅ AirBnB chargé : {len(df_clean)} lignes, {len(colonnes_dispo)} colonnes retenues")
    return df_clean


def generer_airbnb_synthetique():
    
    np.random.seed(42)
    n = 600

    g0 = pd.DataFrame({
        "price": np.random.normal(30, 10, n//3),
        "minimum_nights": np.random.normal(5, 2, n//3),
        "number_of_reviews": np.random.normal(10, 5, n//3),
        "availability_365": np.random.normal(200, 50, n//3)
    })
    g1 = pd.DataFrame({
        "price": np.random.normal(80, 15, n//3),
        "minimum_nights": np.random.normal(2, 1, n//3),
        "number_of_reviews": np.random.normal(50, 15, n//3),
        "availability_365": np.random.normal(150, 40, n//3)
    })
   
    g2 = pd.DataFrame({
        "price": np.random.normal(200, 40, n//3),
        "minimum_nights": np.random.normal(1, 0.5, n//3),
        "number_of_reviews": np.random.normal(80, 20, n//3),
        "availability_365": np.random.normal(300, 60, n//3)
    })

    df = pd.concat([g0, g1, g2], ignore_index=True)
    df = df.clip(lower=0) 
    print(f"✅ Dataset synthétique AirBnB : {len(df)} annonces, 4 variables")
    return df



URL_AIRBNB = "http://data.insideairbnb.com/france/ile-de-france/paris/2023-12-12/visualisations/listings.csv"

df_airbnb = charger_airbnb(URL_AIRBNB)
if df_airbnb is None:
    df_airbnb = generer_airbnb_synthetique()

df_airbnb.describe()

In [ ]:
def choisir_k(X_scaled, k_range=range(2, 9)):
   
    print(f"{'k':>4} | {'Inertie':>10} | {'Silhouette':>10}")
    print("-" * 32)

    resultats = []
    for k in k_range:
        km = KMeans(n_clusters=k, n_init=10, random_state=42)
        km.fit(X_scaled)
        sil = silhouette_score(X_scaled, km.labels_)
        print(f"{k:>4} | {km.inertia_:>10.1f} | {sil:>10.3f}")
        resultats.append((k, km.inertia_, sil))

    meilleur = max(resultats, key=lambda x: x[2])
    print(f"\n→ Meilleur k selon silhouette : k={meilleur[0]} (silhouette={meilleur[2]:.3f})")
    return meilleur[0]


scaler_ab = StandardScaler()
X_airbnb = df_airbnb.values
X_airbnb_scaled = scaler_ab.fit_transform(X_airbnb)

print("=" * 55)
print("  CHOIX DU BON K (avec standardisation)")
print("=" * 55)
k_optimal = choisir_k(X_airbnb_scaled)

In [ ]:
km_final = KMeans(n_clusters=k_optimal, n_init=10, random_state=42)
km_final.fit(X_airbnb_scaled)
df_airbnb["segment"] = km_final.labels_

print("\n=" * 55)
print(f"  PROFIL DES {k_optimal} SEGMENTS")
print("=" * 55)
print(df_airbnb.groupby("segment").mean().round(1).to_string())
print()
print("Taille de chaque segment :")
print(df_airbnb["segment"].value_counts().sort_index().to_string())